# 🚗 TOYOTA SmartLife × EV LifePilot（合併版）
把「油車電車（5 年 TCO 精算 + CSV 統計問答）」與「SmartLife（車主檔案 + 文字回答的 AI 對話）」合成一支程式。

**使用方式**
1. 把兩份 CSV 上傳到 Colab 左側「檔案」（主檔檔名含 `CDP`、補充檔檔名含 `補充`），或在【區塊 0】填入路徑。
2. 左側「密鑰」新增名為 `api` 的 secret（放 Gemini API Key），並開啟「筆記本存取權」。
3. 由上到下依序執行所有儲存格，最後一格會產生 Gradio 網址。

In [1]:
!pip install -q -U google-genai
!pip install -q gradio pandas plotly

import os, re, glob, json, inspect, time
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
from google import genai
from google.genai import types, errors

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 19.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


In [2]:
#=========================================================
# 【區塊 0：參數設定】
# CSV 裡「沒有」的欄位（油價、電價、油耗、電耗）集中放這裡，之後只需要改這一區
# =========================================================
GAS_PRICE_PER_L    = 31.0   # 汽油 NT$/公升
ELEC_PRICE_PER_KWH = 4.5    # 電價 NT$/度
EV_KM_PER_KWH      = 5.8    # bZ4X 每度電可跑公里數
FUEL_KM_PER_L = {"汽油": 12.0, "油電混合": 20.0}  # 各動力型式平均油耗 km/L。汽油沿用原程式；油電混合是估計值，請依官方規格調整
FUEL_KM_PER_L_BY_MODEL = {}  # 想精準到單一車款時在這裡覆寫，例如 {"ALTIS": 14.5, "RAV4 HV": 21.0}

A_CURRENCY_TO_NTD = 1.0     # CSV 金額是「A國幣」(非台幣)。填 1 A國幣 = 幾元台幣；目前先以 1:1 當 NT$ 使用
YEARS = 5                   # TCO 計算年數
MAINT_AGE_BAND = (1, 5)     # 保養費只取「車齡 1~5 年」的車主，口徑與 5 年 TCO 一致（避免老車/新車混在一起比）
MIN_SAMPLE = 30             # 某車款在上述車齡區間內少於此人數 → 改用「同動力型式（汽油/油電混合）車款」的平均
EV_MODEL = "bZ4X"

MAIN_CSV_PATH = None        # CSV 不在目前資料夾時，填完整路徑；None = 自動搜尋
SUPP_CSV_PATH = None

# Gemini API Key 讀取順序：Colab 密鑰 "api" → 下面這個變數 → 環境變數 GEMINI_API_KEY / GOOGLE_API_KEY
# ⚠️ 請不要把真正的金鑰直接寫在筆記本裡再上傳到 GitHub
GEMINI_API_KEY = ""
# 依序嘗試；成功過的模型之後會自動排到最前面。各模型的額度是分開算的，某個額度用完或忙線就會換下一個。
# gemini-2.5-flash 已不開放新用戶（404 no longer available to new users），所以不放；最後一個 -latest 別名會自動指向最新的 Lite 模型
GEMINI_MODELS = ["gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash", "gemini-3.5-flash", "gemini-flash-lite-latest"]

In [3]:
#---------------------------------------------------------
# 【區塊 1：讀取兩份 CSV 檔案】
# 兩份檔案都是 Big5 編碼（不是 UTF-8），欄位名稱還夾有換行，所以要自動判斷編碼 + 正規化欄名
# ---------------------------------------------------------
COLUMN_KEYS = {
    "sex": "性別", "age": "年齡", "city": "居住縣市",
    "visits_1y": "近一年回廠次數",
    "svc_amt_1y": "近一年定保消費金額",
    "repair_cnt_1y": "近一年維修次數",
    "model": "現保有車款",
    "deliver": "現保有車交車日期",
}

def _find_csv(path, must_have, must_not=()):
    if path:
        return path
    hits = []
    for d in (".", "/content"):
        for p in glob.glob(os.path.join(d, "*.csv")):
            name = os.path.basename(p)
            if all(k in name for k in must_have) and not any(k in name for k in must_not):
                hits.append(p)
    if not hits:
        raise FileNotFoundError(f"找不到檔名含 {must_have} 的 CSV，請先上傳到 Colab，或在區塊 0 填入 *_CSV_PATH")
    return max(hits, key=os.path.getsize)

def _read_csv_auto(path):
    for enc in ("utf-8-sig", "big5", "cp950"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"無法判斷 {path} 的編碼")

def _load_cdp(path):
    raw = _read_csv_auto(path)
    raw.columns = [re.sub(r"\s+", "", str(c)) for c in raw.columns]   # 去掉欄名裡的換行/空白
    out = pd.DataFrame()
    for key, kw in COLUMN_KEYS.items():
        hits = [c for c in raw.columns if kw in c]
        if len(hits) != 1:
            raise ValueError(f"{os.path.basename(path)}：找不到唯一欄位「{kw}」，目前欄位：{list(raw.columns)}")
        out[key] = raw[hits[0]]
    for c in ["age", "visits_1y", "svc_amt_1y", "repair_cnt_1y"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    out["deliver"] = pd.to_datetime(out["deliver"], errors="coerce")
    out["model"] = out["model"].fillna("").astype(str).str.strip()
    out = out[~out["model"].isin(["", "nan", "NaN", "None"])].reset_index(drop=True)   # 車款空白的列無法歸類，直接排除
    out["age"] = out["age"].where(out["age"].between(18, 100))        # 0 = 未提供、>100 = 異常值，一律視為缺值
    out["svc_amt_1y"] = out["svc_amt_1y"] * A_CURRENCY_TO_NTD         # A國幣 → 台幣
    return out

df_main = _load_cdp(_find_csv(MAIN_CSV_PATH, ("CDP",), ("補充",)))   # 主檔：各車款車主回廠資料
df_supp = _load_cdp(_find_csv(SUPP_CSV_PATH, ("補充",)))              # 補充檔：bZ4X 車主資料
print(f"✅ CSV 資料庫讀取成功！主檔 {len(df_main):,} 筆、bZ4X 補充檔 {len(df_supp):,} 筆")

✅ CSV 資料庫讀取成功！主檔 573,302 筆、bZ4X 補充檔 3,037 筆


In [4]:
# ---------------------------------------------------------
# 【區塊 2：從 CSV 統計各車款保養費 + 5年 TCO 養車費用預估】
# ---------------------------------------------------------
# 主檔剛好停在 Excel 列數上限(1,048,575)，bZ4X 不完整 → bZ4X 一律用補充檔，其他車款用主檔
df_all = pd.concat([df_main[df_main["model"] != EV_MODEL], df_supp], ignore_index=True)
SNAPSHOT = df_all["deliver"].max()                                    # 資料截止日（以最晚交車日代表）
df_all["years"] = (SNAPSHOT - df_all["deliver"]).dt.days / 365.25     # 截至資料截止日的車齡

lo, hi = MAINT_AGE_BAND

def _powertrain(model):
    m = model.upper()
    if model == EV_MODEL:
        return "純電"
    return "油電混合" if ("HV" in m or m.endswith("-H") or m.startswith("PRIUS")) else "汽油"

def _robust_mean(s, q=0.99):
    s = s.dropna()
    return float(s.clip(upper=s.quantile(q)).mean()) if len(s) else float("nan")   # 蓋掉最高 1% 極端值再平均

df_all["powertrain"] = df_all["model"].map(_powertrain)
_band_ice = df_all[df_all["years"].between(lo, hi) & (df_all["model"] != EV_MODEL)]
POOLED_MAINT = {pt: _robust_mean(g["svc_amt_1y"]) for pt, g in _band_ice.groupby("powertrain")}   # 樣本不足時的備援：同動力型式平均

def _build_model_stats():
    rows = []
    for model, g in df_all.groupby("model"):
        band = g[g["years"].between(lo, hi)]
        enough = len(band) >= MIN_SAMPLE
        rows.append({
            "model": model,
            "powertrain": g["powertrain"].iloc[0],
            "owners": len(g),
            "median_years": g["years"].median(),
            "n_band": len(band),
            "maint_per_year": _robust_mean(band["svc_amt_1y"]) if enough else POOLED_MAINT[g["powertrain"].iloc[0]],
            "maint_fallback": not enough,
            "visits_band": _robust_mean(band["visits_1y"]) if enough else float("nan"),
            "repair_band": _robust_mean(band["repair_cnt_1y"]) if enough else float("nan"),
        })
    return pd.DataFrame(rows).set_index("model")

MODEL_STATS = _build_model_stats()
EV_STATS  = MODEL_STATS.loc[EV_MODEL]
ICE_STATS = MODEL_STATS.drop(index=EV_MODEL)
CAR_CHOICES = ICE_STATS[ICE_STATS["owners"] >= MIN_SAMPLE].sort_values("owners", ascending=False).index.tolist()

def calculate_tco(current_car_type, annual_km, years=YEARS):
    s = ICE_STATS.loc[current_car_type]
    km_per_l = FUEL_KM_PER_L_BY_MODEL.get(current_car_type, FUEL_KM_PER_L[s["powertrain"]])

    ice_energy_cost = (annual_km / km_per_l) * GAS_PRICE_PER_L * years
    ev_energy_cost  = (annual_km / EV_KM_PER_KWH) * ELEC_PRICE_PER_KWH * years
    ice_maint_cost  = s["maint_per_year"] * years          # ← 來自 CSV
    ev_maint_cost   = EV_STATS["maint_per_year"] * years   # ← 來自 CSV（bZ4X 補充檔）

    total_ice = ice_energy_cost + ice_maint_cost
    total_ev  = ev_energy_cost + ev_maint_cost
    saved_money = total_ice - total_ev
    detail = {
        "km_per_l": km_per_l, "powertrain": s["powertrain"],
        "ice_maint_yr": s["maint_per_year"], "ice_n": int(s["n_band"]), "ice_fallback": bool(s["maint_fallback"]),
        "ev_maint_yr": EV_STATS["maint_per_year"], "ev_n": int(EV_STATS["n_band"]),
        "ice_energy": ice_energy_cost, "ev_energy": ev_energy_cost,      # 5 年能源費（給 AI 顧問引用）
        "ice_maint": ice_maint_cost, "ev_maint": ev_maint_cost,          # 5 年保養費（給 AI 顧問引用）
    }
    return total_ice, total_ev, saved_money, detail

In [5]:
# ---------------------------------------------------------
# 【區塊 3：Gemini AI 顧問】
# 100 萬列 CSV 不能整份塞進 prompt，改餵「CSV 統計摘要」+「該車主的檔案與精算結果」
# AI 顧問只用文字回答（不產生圖表）
# ---------------------------------------------------------
def _load_api_key():
    try:
        from google.colab import userdata
        key = userdata.get("api")
        if key:
            return key
    except Exception:
        pass
    return GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY") or ""

API_KEY = _load_api_key()
client = genai.Client(api_key=API_KEY) if API_KEY else None   # 沒有金鑰也能先開介面，對話時會提示

def _fmt(x, nd=1):
    return "樣本不足" if pd.isna(x) else f"{x:.{nd}f}"   # 不加千分位，避免和 CSV 的逗號分隔混淆

def build_knowledge_text():
    ev = df_supp
    sex = ev["sex"].where(ev["sex"].isin(["男", "女"]), "其他/未提供").value_counts(normalize=True)
    top_city = ev.loc[ev["city"] != "99_other", "city"].value_counts().head(5)
    by_year = ev["deliver"].dt.year.value_counts().sort_index()
    db1 = (
        f"bZ4X 車主補充資料（共 {len(ev):,} 位，資料截止日 {SNAPSHOT:%Y-%m-%d}）：\n"
        f"- 性別：{'、'.join(f'{k}{v:.0%}' for k, v in sex.items())}；年齡中位數 {ev['age'].median():.0f} 歲\n"
        f"- 居住縣市前五名：{'、'.join(f'{k}{v:,}人' for k, v in top_city.items())}\n"
        f"- 交車年度：{'、'.join(f'{int(k)}年{v:,}人' for k, v in by_year.items())}\n"
        f"- 近一年平均回廠 {_robust_mean(ev['visits_1y']):.1f} 次、平均維修 {_robust_mean(ev['repair_cnt_1y']):.1f} 次"
    )
    lines = [f"車款,動力,車主數,車齡中位數(年),車齡{lo}-{hi}年樣本數,車齡{lo}-{hi}年_年均定保花費(NT$),車齡{lo}-{hi}年_近一年平均回廠次數,車齡{lo}-{hi}年_近一年平均維修次數"]
    for m, r in MODEL_STATS.sort_values("owners", ascending=False).iterrows():
        lines.append(f"{m},{r['powertrain']},{r['owners']},{r['median_years']:.1f},{r['n_band']},"
                     f"{'樣本不足' if r['maint_fallback'] else _fmt(r['maint_per_year'], 0)},{_fmt(r['visits_band'])},{_fmt(r['repair_band'])}")
    db2 = "各車款車主回廠統計（主檔 {:,} 筆彙整）：\n".format(len(df_main)) + "\n".join(lines)
    return db1, db2

DB1_TEXT, DB2_TEXT = build_knowledge_text()

SYSTEM_RULES = """
你是 TOYOTA 專屬的「SmartLife 智能隨行顧問 × EV LifePilot 油轉電 AI 顧問」。
你的任務是消除車主對電動車 (bZ4X) 的疑慮，並協助規劃用車成本。
請使用繁體中文，語氣親切、專業、有同理心，善用白話比喻。

【資料使用規則】
1. 請【嚴格根據】下方兩份資料庫，以及每次提問附上的「車主檔案」與「系統精算結果」回答；引用 CSV 數字時請說明是「車主回廠資料統計」。
2. 資料庫沒有的內容（例如電池保固年限、充電樁安裝規定與補助、車價、原廠優惠）請誠實告知資料庫未提供，
   只給一般性的建議，並請客戶向 TOYOTA 經銷商確認，不可編造數字、保固條款或確切價格。
3. 費用試算的油價、電價、油耗、電耗為「假設值」，回答時須說明假設，並提醒實際以原廠報價為準。

【輸出格式】
請直接以文字回答，可用 Markdown 條列。不要輸出 JSON、程式碼或圖表；
若車主要求畫圖表，請改用文字（條列或簡短比較）說明數字即可。
"""

SYSTEM_INSTRUCTION = (
    SYSTEM_RULES
    + f"\n金額原始單位為 A 國幣，本系統以 1 A國幣 = {A_CURRENCY_TO_NTD:g} 元新台幣換算後呈現。\n"
    + f"\n【資料庫 1】\n{DB1_TEXT}\n\n【資料庫 2】\n{DB2_TEXT}\n"
)

_MODEL_ORDER = list(GEMINI_MODELS)
_RETRY_CODES = (500, 503, 504)   # 伺服器暫時忙線：同一個模型等 2 秒再試一次；429（額度用完）等其他錯誤直接換下一個模型

def _err_brief(e):
    if isinstance(e, errors.APIError):
        return f"{e.code} {e.status or ''} {(e.message or '')[:160]}".strip()
    return f"{type(e).__name__}: {str(e)[:160]}"

def _ask_gemini(system_instruction, prompt):
    if client is None:
        raise ValueError("尚未設定 Gemini API Key：請在 Colab 左側「密鑰」新增名為 api 的 secret，或在區塊 0 填入 GEMINI_API_KEY")
    failures = []                                            # 每個模型各自的失敗原因都留下來，不要只剩最後一個
    for name in list(_MODEL_ORDER):
        for attempt in range(2):
            try:
                resp = client.models.generate_content(
                    model=name, contents=prompt,
                    config=types.GenerateContentConfig(
                        system_instruction=system_instruction))
            except Exception as e:
                if attempt == 0 and isinstance(e, errors.APIError) and e.code in _RETRY_CODES:
                    time.sleep(2)
                    continue
                failures.append(f"{name}：{_err_brief(e)}")
                break
            if name != _MODEL_ORDER[0]:                      # 記住可用的模型，下次直接用
                _MODEL_ORDER.remove(name)
                _MODEL_ORDER.insert(0, name)
            return resp.text or ""
    raise RuntimeError("所有 Gemini 模型都無法使用：\n" + "\n".join(f"- {f}" for f in failures))

# 取出 Chatbot 歷史訊息中的純文字（不同 Gradio 版本 content 可能是 str / list / dict / 元件）
def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(c.get("text", "") if isinstance(c, dict) else (c if isinstance(c, str) else "") for c in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return ""   # 其他元件不放進文字紀錄

def tco_context(car, annual_km):
    """把目前車款 + 年里程的精算結果寫成文字，讓 AI 顧問的數字和儀表板一致。"""
    if car == EV_MODEL:
        return f"車主目前已經駕駛 {EV_MODEL}（車主資料 {int(EV_STATS['owners']):,} 位），不需要換購試算。"
    if car not in ICE_STATS.index:
        return "（車款不在資料庫內，無法精算）"
    ice, ev, saved, d = calculate_tco(car, annual_km)
    src = (f"{car} 車齡{lo}-{hi}年樣本不足，保養費採同為{d['powertrain']}車款的平均"
           if d["ice_fallback"] else f"{car} 車齡{lo}-{hi}年車主 {d['ice_n']:,} 位")
    return (
        f"- 假設：年行駛 {annual_km:,} 公里、{YEARS} 年；汽油 NT${GAS_PRICE_PER_L:g}/L、{car} 約 {d['km_per_l']:g} km/L；"
        f"電價 NT${ELEC_PRICE_PER_KWH:g}/度、{EV_MODEL} 約 {EV_KM_PER_KWH:g} km/kWh\n"
        f"- 現有 {car}（{d['powertrain']}）{YEARS} 年：能源 NT${d['ice_energy']:,.0f} + 定保 NT${d['ice_maint']:,.0f}"
        f"（每年 NT${d['ice_maint_yr']:,.0f}）= 總計 NT${ice:,.0f}\n"
        f"- 換購 {EV_MODEL} {YEARS} 年：能源 NT${d['ev_energy']:,.0f} + 定保 NT${d['ev_maint']:,.0f}"
        f"（每年 NT${d['ev_maint_yr']:,.0f}）= 總計 NT${ev:,.0f}\n"
        f"- 結論：換購 {EV_MODEL} {YEARS} 年{'可省下' if saved >= 0 else '約多花'} NT${abs(saved):,.0f}\n"
        f"- 定保資料依據：{src}；{EV_MODEL} 車齡{lo}-{hi}年車主 {d['ev_n']:,} 位"
    )

def build_prompt(user_message, history, name, car, mileage, annual_km, pref):
    lines = []
    for m in history[-12:]:                                  # 只帶最近 12 則，避免 prompt 越來越長
        if not isinstance(m, dict) or (m.get("metadata") or {}).get("status") == "pending":   # 略過「顧問思考中」暫時訊息
            continue
        t = _text(m.get("content"))
        if t and not t.startswith("💡 顧問小提醒"):
            lines.append(f"{'車主' if m.get('role') == 'user' else '顧問'}：{t}")
    return (
        f"【車主檔案】\n"
        f"- 稱呼：{name or '（未填）'}\n- 目前車款：{car}\n- 目前里程數：{mileage or '（未填）'}\n"
        f"- 預估年行駛里程：{annual_km:,} 公里\n- 用車習慣/喜好：{pref or '（未填）'}\n\n"
        f"【系統精算結果】\n{tco_context(car, annual_km)}\n\n"
        f"--- 歷史對話紀錄 ---\n" + "\n".join(lines) + f"\n\n車主：{user_message}\n請以文字回答："
    )

CHAT_PLACEHOLDER   = "請輸入您的問題並按 Enter 送出..."
LOCKED_PLACEHOLDER = "請先在上方點擊「確認並建立檔案」..."
BUSY_PLACEHOLDER   = "顧問思考中，請稍候..."

def _thinking_message():
    # metadata.status="pending" 會讓 Gradio 對話框顯示轉圈圈的「思考中」氣泡，Gemini 回覆後整則被答案取代
    return {"role": "assistant", "content": "正在參考您的車主檔案與車主回廠資料統計，整理回覆中…",
            "metadata": {"title": "🤔 顧問思考中", "status": "pending"}}

def sync_chat_box(ready, busy):
    """輸入框 / 送出鈕的狀態只由兩個狀態推導：檔案是否已建立(ready)、顧問是否正在回覆(busy)。
    各處不再各改各的，才不會出現「回覆中又按確認 → 輸入框被解鎖 → 對話錯亂」這類互相打架的情形。"""
    if busy:
        return gr.update(interactive=False, placeholder=BUSY_PLACEHOLDER), gr.update(interactive=False)
    if ready:
        return gr.update(interactive=True, placeholder=CHAT_PLACEHOLDER), gr.update(interactive=True)
    return gr.update(interactive=False, placeholder=LOCKED_PLACEHOLDER), gr.update(interactive=False)

def chat_with_ai(user_message, history, name, car, mileage, annual_km, pref, ready):
    # outputs = [輸入框, 對話框, 送出鈕, busy 狀態]；回覆結束後由 sync_chat_box 負責解鎖
    history = list(history or [])
    if not ready:                                            # 正常操作碰不到（輸入框是鎖住的），純粹保險
        gr.Warning("請先建立車主檔案")
        yield gr.update(), history, gr.update(), gr.skip()
        return
    if not user_message or not user_message.strip():
        yield gr.update(), history, gr.update(), gr.skip()
        return

    prompt = build_prompt(user_message, history, name, car, mileage, int(annual_km or 0), pref)
    history.append({"role": "user", "content": user_message})
    # 先顯示使用者訊息 +「顧問思考中」；等待期間鎖住輸入框與送出鈕，避免連續送出讓對話錯亂
    yield (gr.update(value="", interactive=False, placeholder=BUSY_PLACEHOLDER),
           history + [_thinking_message()], gr.update(interactive=False), True)

    try:
        answer = _ask_gemini(SYSTEM_INSTRUCTION, prompt).strip()
        history.append({"role": "assistant", "content": answer or "（顧問暫時沒有產生回覆，請換個問法再試一次）"})
    except Exception as e:
        history.append({"role": "assistant", "content": f"💡 顧問小提醒：發生異常，請確認 API Key、額度或資料庫狀態。\n\n{e}"})
    yield gr.update(), history, gr.update(), False

In [6]:
# ---------------------------------------------------------
# 【區塊 4：車主檔案 + TCO 儀表板更新函數】
# ---------------------------------------------------------
def confirm_profile(name, car, mileage, annual_km, pref):
    # outputs = [系統狀態, ready 狀態]；輸入框要不要解鎖交給 sync_chat_box 決定（回覆中不會被誤解鎖）
    if not name or not name.strip() or not car:
        return "⚠️ 請至少填寫您的稱呼與車款！", False
    if car == EV_MODEL:
        info = f"{EV_MODEL}（純電，車主 {int(EV_STATS['owners']):,} 位）"
    else:
        r = ICE_STATS.loc[car]
        info = f"{car}（{r['powertrain']}，車主 {int(r['owners']):,} 位）"
    return (f"✅ 檔案建立成功！您好，{name.strip()}。\n系統已紀錄您的愛車為 {info}、年行駛 {int(annual_km):,} 公里，請在下方開始對話。", True)

def update_dashboard(car_type, annual_km):
    annual_km = int(annual_km)
    if car_type == EV_MODEL:
        return None, f"🎉 您目前駕駛的就是 **{EV_MODEL}**，不需要換購試算！想了解用車成本或保養，直接問下方的 AI 顧問吧。"
    if car_type not in ICE_STATS.index:
        return None, "⚠️ 請先選擇目前駕駛的車款。"

    ice_cost, ev_cost, saved, d = calculate_tco(car_type, annual_km)

    fig = go.Figure(data=[
        go.Bar(name='現有燃油車', x=[f'{YEARS}年總花費(能源+保養)'], y=[ice_cost], marker_color='#E53935'),
        go.Bar(name='TOYOTA bZ4X', x=[f'{YEARS}年總花費(能源+保養)'], y=[ev_cost], marker_color='#4CAF50')
    ])

    fig.update_layout(
        title=f"<b>換購 bZ4X {YEARS} 年預估{'可為您省下' if saved >= 0 else '約多花'} NT$ {abs(saved):,.0f} 元</b>",
        barmode='group',
        height=380,
        margin=dict(l=20, r=20, t=50, b=20)
    )

    summary_text = (
        f"💡 **專屬診斷報告**：依您駕駛 **{car_type}**、每年行駛 **{annual_km:,}** 公里計算，"
        f"換購 bZ4X {YEARS} 年累積{'可省下' if saved >= 0 else '反而多花'}約 **NT$ {abs(saved):,.0f}** 元的油資與保養總支出！"
    )
    ice_src = (f"{car_type} 車齡{lo}-{hi}年樣本不足，採同為{d['powertrain']}車款的平均" if d["ice_fallback"]
               else f"{car_type} 車齡{lo}-{hi}年車主 {d['ice_n']:,} 位")
    summary_text += (
        f"\n\n📊 **資料依據（CSV）**：{ice_src}，每年定保平均 NT$ {d['ice_maint_yr']:,.0f}；"
        f"bZ4X 車主 {d['ev_n']:,} 位，每年定保平均 NT$ {d['ev_maint_yr']:,.0f}。"
        f"能源費用為假設值：{d['km_per_l']:g} km/L × 汽油 NT$ {GAS_PRICE_PER_L:g}/L；"
        f"{EV_KM_PER_KWH:g} km/kWh × 電價 NT$ {ELEC_PRICE_PER_KWH:g}/度。"
    )
    return fig, summary_text

In [ ]:
# ---------------------------------------------------------
# 【區塊 5：建立 Gradio Web UI】
# 步驟 1 建立車主檔案 → 步驟 2 看 5 年 TCO → 步驟 3 與 AI 顧問對話（文字回答）
# ---------------------------------------------------------
GR_MAJOR = int(gr.__version__.split(".")[0])
BLOCKS_KW = {} if GR_MAJOR >= 6 else {"theme": gr.themes.Soft()}     # Gradio 6 起 theme 改放 launch()
LAUNCH_KW = {"theme": gr.themes.Soft()} if GR_MAJOR >= 6 else {}

PROFILE_CARS = CAR_CHOICES + [EV_MODEL]    # 已經是 bZ4X 的車主也能建立檔案、問顧問（儀表板會提示不需換購試算）
EXAMPLE_QUESTIONS = ["大樓沒辦法裝充電樁怎麼辦？",
                     "bZ4X 的保養費用和我現在的車比起來如何？",
                     "幫我算出接下來 5 年的累積用車成本，油車和 bZ4X 比一比",
                     "bZ4X 電池保固多久？"]
# 回覆完成後把游標放回輸入框；但使用者若已點去別的欄位（例如正在填檔案）就不搶焦點，也不捲動頁面
FOCUS_INPUT_JS = ("() => { setTimeout(() => { const t = document.querySelector('#msg_input textarea, #msg_input input');"
                  " const a = document.activeElement;"
                  " const free = !a || a === document.body || !!a.closest('#msg_input') || (a.tagName === 'BUTTON' && a.textContent.includes('送出'));"
                  " if (t && !t.disabled && free) t.focus({preventScroll: true}); }, 150); }")

def build_app():
    with gr.Blocks(title="TOYOTA SmartLife × EV LifePilot", **BLOCKS_KW) as demo:
        gr.Markdown("# 🚗 TOYOTA SmartLife × EV LifePilot - 個人化油轉電 AI 隨行顧問")
        gr.Markdown("先建立您的專屬檔案，AI 會為您精算 5 年持有成本，並以車主回廠資料回答您的購車與充電疑慮。")

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### 📋 步驟 1：建立車主檔案")
                name_input = gr.Textbox(label="您的稱呼", placeholder="例如：魏先生")
                car_input = gr.Dropdown(PROFILE_CARS, label="目前駕駛車款", value=PROFILE_CARS[0])
                mileage_input = gr.Textbox(label="目前大約里程數", placeholder="例如：23000")
                km_input = gr.Slider(minimum=5000, maximum=50000, step=1000, label="預估年行駛里程 (公里)", value=15000)
                pref_input = gr.Textbox(label="您的用車習慣與喜好", placeholder="例如：每天通勤")
                confirm_btn = gr.Button("✅ 確認並建立檔案", variant="primary")
                status_text = gr.Textbox(label="系統狀態", interactive=False)

            with gr.Column(scale=1):
                gr.Markdown("### 📊 步驟 2：5 年 TCO 養車成本精算")
                calc_btn = gr.Button("⚡ 立即精算 5 年 TCO 養車成本")
                output_text = gr.Markdown("建立檔案或點擊上方按鈕獲取診斷分析報告...")
                chart_output = gr.Plot(label="5 年持有成本比較圖")

        gr.Markdown("---")
        gr.Markdown("### 💬 步驟 3：24H 購車與充電疑慮諮詢 (GenAI 隨行顧問)")
        try:
            chatbot = gr.Chatbot(label="顧問回覆", height=500, type="messages")
        except TypeError:   # 新版 Gradio 已移除 type 參數，預設即 messages 格式
            chatbot = gr.Chatbot(label="顧問回覆", height=500)

        ready = gr.State(False)   # 車主檔案是否已建立
        busy = gr.State(False)    # 顧問是否正在回覆
        with gr.Row():
            msg_input = gr.Textbox(
                label="請輸入您的問題", placeholder=LOCKED_PLACEHOLDER,
                scale=4, interactive=False, elem_id="msg_input"
            )
            submit_btn = gr.Button("📤 送出", scale=1, interactive=False)

        gr.Markdown("試試看這些問法（點一下會填入輸入框）：")
        with gr.Row():
            for q in EXAMPLE_QUESTIONS:   # 只填文字，不碰輸入框的鎖定狀態（gr.Examples 會把它重設）
                gr.Button(q, size="sm").click(lambda q=q: q, None, msg_input, queue=False)

        # 建立檔案 → 順便跑一次 TCO 精算；之後改了車款/里程可再按「立即精算」
        confirm_btn.click(confirm_profile,
                          inputs=[name_input, car_input, mileage_input, km_input, pref_input],
                          outputs=[status_text, ready]
                          ).then(sync_chat_box, inputs=[ready, busy], outputs=[msg_input, submit_btn]
                          ).then(update_dashboard, inputs=[car_input, km_input], outputs=[chart_output, output_text])
        calc_btn.click(update_dashboard, inputs=[car_input, km_input], outputs=[chart_output, output_text])

        # 回覆期間輸入框與送出鈕會被鎖住（顯示「顧問思考中」），回覆完成後解鎖並把游標放回輸入框
        # Enter 與送出鈕合併成同一個事件(gr.on)，才不會在回覆期間被各送一次；concurrency_limit 讓多位使用者同時問時各自都看得到「思考中」
        chat_inputs = [msg_input, chatbot, name_input, car_input, mileage_input, km_input, pref_input, ready]
        chat_outputs = [msg_input, chatbot, submit_btn, busy]
        gr.on(triggers=[msg_input.submit, submit_btn.click], fn=chat_with_ai,
              inputs=chat_inputs, outputs=chat_outputs, concurrency_limit=8
              ).then(sync_chat_box, inputs=[ready, busy], outputs=[msg_input, submit_btn]
              ).then(None, None, None, js=FOCUS_INPUT_JS)

    return demo

if __name__ == "__main__":
    demo = build_app()
    demo.launch(share=True, debug=True, **LAUNCH_KW)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d4000c034d1d395877.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
